# Proyek Akhir Big Data & Data Analytics
## Analisis Pasar dan Pemetaan Daya Saing Akomodasi Digital Airbnb di Provinsi Lampung: Studi Komparatif Kabupaten Lampung Tengah

**Disusun untuk:** Laporan Tugas Akhir Praktikum Big Data & Data Analytics  
**Fokus Penelitian:** Evaluasi pasar akomodasi Airbnb di Provinsi Lampung dengan analisis mendalam terhadap posisi daya saing Kabupaten Lampung Tengah.  
**Pipeline:** Extract, Transform, Load (ETL) menggunakan Python, Pemodelan Data Warehouse di Google BigQuery, dan Visualisasi Business Intelligence di Looker Studio.

---

### Deskripsi Modul ETL Ini:
Notebook ini menyajikan pipeline ETL komprehensif yang dirancang untuk mengintegrasikan data pasar Airbnb per kabupaten/kota di Provinsi Lampung, melakukan standarisasi data, pembersihan data (data cleaning), rekayasa fitur bisnis (feature engineering), hingga ekspor dataset bersih serta skema data yang siap dimuat ke **Google BigQuery**.


## 0. Setup Environment & Import Library

Pada tahap ini, kita mengimpor seluruh pustaka yang dibutuhkan:
- `pandas` & `numpy` untuk manipulasi dan transformasi data tabel.
- `glob` & `os` untuk traversal direktori dan berkas dataset.
- `json` untuk ekspor skema metadata Google BigQuery.


In [ ]:
# Setup pustaka dan konfigurasi lingkungan
import os
import glob
import json
import numpy as np
import pandas as pd

# Pengaturan tampilan pandas agar informatif
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Environment berhasil disiapkan.")


## 1. Extract & Merge (Integrasi Data Multi-Sumber)

### Konteks Data:
Pada folder `datasets/`, terdapat 12 berkas CSV yang mewakili hasil pencarian listing Airbnb di masing-masing kabupaten/kota di Provinsi Lampung, serta 1 berkas gabungan `AirBnB_Lampung_All.csv`.

### Aturan Data Engineering:
1. Kita mengecualikan `AirBnB_Lampung_All.csv` saat penggabungan agar data tidak terduplikasi dua kali.
2. Kita membaca ke-12 berkas per wilayah, mengambil label nama wilayah aktual, dan menggabungkannya secara dinamis.
3. Kita melakukan validasi jumlah baris total gabungan terhadap berkas `AirBnB_Lampung_All.csv`.


In [ ]:
# Deteksi berkas per wilayah di folder datasets
data_dir = '../datasets' if os.path.exists('../datasets') else 'datasets'
all_files = sorted(glob.glob(os.path.join(data_dir, '*.csv')))

# Saring hanya 12 file kabupaten/kota (kecualikan file gabungan Lampung All)
regional_files = [f for f in all_files if 'Lampung_All' not in os.path.basename(f)]

print(f"Total file regional ditemukan: {len(regional_files)} file\n")

dfs = []
audit_list = []

for fpath in regional_files:
    fname = os.path.basename(fpath)
    df_temp = pd.read_csv(fpath)
    wilayah_val = df_temp['wilayah'].iloc[0] if 'wilayah' in df_temp.columns and len(df_temp) > 0 else 'Unknown'
    audit_list.append({
        'Nama File': fname,
        'Wilayah': wilayah_val,
        'Jumlah Baris': len(df_temp),
        'Jumlah Kolom': len(df_temp.columns)
    })
    dfs.append(df_temp)

df_audit = pd.DataFrame(audit_list)
display(df_audit)

# Gabungkan seluruh dataframe regional
df_raw = pd.concat(dfs, ignore_index=True)
initial_rows = len(df_raw)
print(f"\nTotal baris hasil gabungan 12 berkas: {initial_rows} baris.")

# Validasi terhadap berkas AirBnB_Lampung_All.csv
all_path = os.path.join(data_dir, 'AirBnB_Lampung_All.csv')
if os.path.exists(all_path):
    df_all_ref = pd.read_csv(all_path)
    print(f"Total baris berkas referensi AirBnB_Lampung_All.csv: {len(df_all_ref)} baris.")
    assert initial_rows == len(df_all_ref), "Peringatan: Jumlah baris tidak cocok!"
    print(">> VALIDASI SUKSES: Jumlah baris gabungan 100% konsisten!")


### Inspeksi Struktur Data Mentah & Sampel Baris

Mari kita periksa tipe data awal, missing values, dan sampel data khususnya untuk **Kabupaten Lampung Tengah**.


In [ ]:
# Tampilkan 5 baris pertama data gabungan
print("--- 5 Sampel Baris Teratas Dataset Mentah ---")
display(df_raw.head())

# Tampilkan informasi tipe data dan missing values awal
print("\n--- Informasi Struktur Kolom Dataset Mentah ---")
df_raw.info()

# Tampilkan data khusus Kabupaten Lampung Tengah
print("\n--- Seluruh Data Mentah Kabupaten Lampung Tengah ---")
display(df_raw[df_raw['wilayah'] == 'Kabupaten Lampung Tengah'])


## 2. Data Cleaning & Standarisasi (BigQuery Standard)

Pada tahap ini, kita melakukan serangkaian pembersihan data:
1. **Standarisasi Nama Kolom:** Mengubah semua nama kolom menjadi format `snake_case` huruf kecil tanpa spasi dan karakter khusus, serta menamai ulang kolom `wilayah` menjadi `kabupaten_kota`.
2. **Pembersihan Whitespace:** Menghapus kelebihan spasi di awal dan akhir seluruh data teks/string.
3. **Pembersihan dan Validasi Kolom Harga (`harga`):**
   - Format asli mengandung simbol `"Rp"`, pemisah ribuan `","`, dan spasi (contoh: `"Rp 1,957,119"`).
   - Kita bersihkan simbol tersebut dan ubah menjadi bilangan bulat (`int64`).
   - Melakukan filter harga anomali ($\le 0$ atau bernilai null).
4. **Pembersihan Kolom Rating (`rating`):**
   - Rating berisi nilai numerik (`5.0`, `4.94`), teks `'New'` (listing yang baru terdaftar tanpa ulasan), serta `NaN`.
   - Kita buat penanda boolean `is_new_listing` untuk membedakan listing baru.
   - Kolom rating diubah ke numerik `float64`, di mana `'New'` dan missing values diset sebagai `NaN` (standar BigQuery `NULL`).
5. **Pembersihan Kolom Ulasan (`jumlah_review`):**
   - Mengonversi data ulasan ke bilangan bulat (`int64`) dan mengisi null dengan 0.
6. **Penanganan Baris Duplikat:**
   - Memeriksa duplikasi baris persis (`exact duplicates`) dan mendokumentasikan metrik sebelum vs sesudah.


In [ ]:
# 1. Standarisasi nama kolom ke snake_case BigQuery
df_clean = df_raw.copy()
df_clean.columns = [c.strip().lower().replace(' ', '_') for c in df_clean.columns]
if 'wilayah' in df_clean.columns:
    df_clean = df_clean.rename(columns={'wilayah': 'kabupaten_kota'})

# 2. Whitespace trimming pada semua kolom teks
for col in df_clean.select_dtypes(include=['object', 'string']).columns:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# 3. Dokumentasi Missing Values Sebelum Pembersihan
missing_before = df_clean.isnull().sum()

# 4. Pengecekan Duplikat Baris Persis (Exact Duplicates)
dup_exact_count = df_clean.duplicated().sum()
print(f"Jumlah baris duplikat persis sebelum cleaning: {dup_exact_count} baris")
if dup_exact_count > 0:
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# 5. Pembersihan Kolom Harga
df_clean['harga'] = (
    df_clean['harga']
    .str.replace('Rp', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df_clean['harga'] = pd.to_numeric(df_clean['harga'], errors='coerce')

# Filter anomali harga (<= 0 atau kosong)
anomali_harga = (df_clean['harga'] <= 0) | (df_clean['harga'].isna())
print(f"Jumlah harga anomali (<= 0 atau null): {anomali_harga.sum()} baris")
df_clean = df_clean[~anomali_harga].copy()
df_clean['harga'] = df_clean['harga'].astype('int64')

# 6. Pembersihan Kolom Rating & Identifikasi Listing Baru
df_clean['is_new_listing'] = df_clean['rating'].str.lower() == 'new'

df_clean['rating'] = pd.to_numeric(
    df_clean['rating'].replace({'New': np.nan, 'N/A': np.nan, 'nan': np.nan, 'None': np.nan}),
    errors='coerce'
)

# 7. Pembersihan Kolom Jumlah Review
df_clean['jumlah_review'] = pd.to_numeric(df_clean['jumlah_review'], errors='coerce').fillna(0).astype('int64')

# 8. Tabel Evaluasi Missing Values Sebelum vs Sesudah Cleaning
missing_after = df_clean.isnull().sum()
df_missing_eval = pd.DataFrame({
    'Missing Sebelum': missing_before,
    'Missing Sesudah': missing_after,
    'Tipe Data Final': df_clean.dtypes
})
print("\n--- Tabel Perbandingan Missing Values Sebelum vs Sesudah Cleaning ---")
display(df_missing_eval)


## 3. Feature Engineering (Nilai Tambah Bisnis)

Untuk memaksimalkan insight komparatif dan kemudahan visualisasi pada **Google Looker Studio**, kita menambahkan sejumlah fitur turunan:
1. **`listing_id`:** Ekstraksi ID unik properti langsung dari tautan URL (`link`), e.g., `748970771910197648`.
2. **`tipe_properti`, `lokasi_spesifik`, `judul_listing`:** Hasil parsing terstruktur dari pola kolom `nama` (`<tipe_properti> in <lokasi_spesifik> - <judul_listing>`).
3. **`wilayah_fokus`:** Klasifikasi biner untuk studi komparatif:
   - `"Kabupaten Lampung Tengah"`
   - `"Kabupaten/Kota Lain (Lampung)"`
4. **`price_tier`:** Segmentasi tarif sewa per malam berbasis kuartil distribusi harga:
   - **Budget:** Tarif $\le Q_1$ ($\le 	ext{Rp } 1.955.692$)
   - **Mid-Range:** Tarif antara $Q_1$ dan $Q_3$ ($	ext{Rp } 1.955.693 - 	ext{Rp } 5.497.506$)
   - **Premium:** Tarif $> Q_3$ ($> 	ext{Rp } 5.497.506$)
5. **`has_reviews`:** Penanda boolean apakah listing telah memiliki ulasan tamu (`jumlah_review > 0`).
6. **`rating_kategori`:** Kategorisasi ulasan (`"Sangat Bagus (4.8 - 5.0)"`, `"Bagus (4.5 - 4.79)"`, `"Cukup (< 4.5)"`, `"Listing Baru (New)"`, `"Tanpa Rating"`).


In [ ]:
# 1. Ekstraksi listing_id dari URL link
df_clean['listing_id'] = df_clean['link'].apply(
    lambda x: str(x).split('rooms/')[-1].split('?')[0].strip()
)

# 2. Parsing kolom nama menjadi komponen terstruktur
def parse_nama_listing(name_str):
    prop_type, loc, title = 'Other', 'Lampung', name_str
    if ' in ' in name_str:
        parts = name_str.split(' in ', 1)
        prop_type = parts[0].strip()
        rest = parts[1]
        if ' - ' in rest:
            loc_parts = rest.split(' - ', 1)
            loc = loc_parts[0].strip()
            title = loc_parts[1].strip()
        else:
            loc = rest.strip()
            title = rest.strip()
    elif ' - ' in name_str:
        parts = name_str.split(' - ', 1)
        loc = parts[0].strip()
        title = parts[1].strip()
    return pd.Series([prop_type, loc, title])

df_clean[['tipe_properti', 'lokasi_spesifik', 'judul_listing']] = df_clean['nama'].apply(parse_nama_listing)

# 3. Rekayasa fitur wilayah_fokus (Studi Komparatif Lampung Tengah)
df_clean['wilayah_fokus'] = np.where(
    df_clean['kabupaten_kota'] == 'Kabupaten Lampung Tengah',
    'Kabupaten Lampung Tengah',
    'Kabupaten/Kota Lain (Lampung)'
)

# 4. Rekayasa fitur price_tier berdasarkan Kuartil Harga
q1 = df_clean['harga'].quantile(0.25)
q3 = df_clean['harga'].quantile(0.75)
print(f"Ambang Batas Kuartil Harga:")
print(f"  * Kuartil 1 (Q1) : Rp {q1:,.0f} -> Batas Budget")
print(f"  * Kuartil 3 (Q3) : Rp {q3:,.0f} -> Batas Premium")

def hitung_price_tier(harga):
    if harga <= q1:
        return 'Budget'
    elif harga <= q3:
        return 'Mid-Range'
    else:
        return 'Premium'

df_clean['price_tier'] = df_clean['harga'].apply(hitung_price_tier)

# 5. Rekayasa fitur has_reviews
df_clean['has_reviews'] = df_clean['jumlah_review'] > 0

# 6. Rekayasa fitur rating_kategori
def tentukan_kategori_rating(row):
    r = row['rating']
    if pd.isna(r):
        if row['is_new_listing']:
            return 'Listing Baru (New)'
        return 'Tanpa Rating'
    elif r >= 4.8:
        return 'Sangat Bagus (4.8 - 5.0)'
    elif r >= 4.5:
        return 'Bagus (4.5 - 4.79)'
    else:
        return 'Cukup (< 4.5)'

df_clean['rating_kategori'] = df_clean.apply(tentukan_kategori_rating, axis=1)

# Susun ulang kolom akhir secara sistematis
final_cols = [
    'listing_id',
    'nama',
    'tipe_properti',
    'lokasi_spesifik',
    'judul_listing',
    'kabupaten_kota',
    'wilayah_fokus',
    'harga',
    'rating',
    'rating_kategori',
    'is_new_listing',
    'jumlah_review',
    'has_reviews',
    'price_tier',
    'link'
]

df_final = df_clean[final_cols].copy()
print(f"Dimensi akhir dataset: {df_final.shape[0]} baris x {df_final.shape[1]} kolom.")
display(df_final.head(5))


## 4. Analisis Eksploratif & Validasi Komparatif

Mari kita lakukan verifikasi analitik terhadap metrik-metrik kunci sebelum data dimuat ke Data Warehouse.


In [ ]:
# 1. Distribusi Listing per Kabupaten/Kota
print("=== DISTRIBUSI LISTING PER KABUPATEN/KOTA ===")
dist_wilayah = df_final['kabupaten_kota'].value_counts().reset_index()
dist_wilayah.columns = ['Kabupaten/Kota', 'Total Listing']
display(dist_wilayah)

# 2. Studi Komparatif: Lampung Tengah vs Wilayah Lain
print("\n=== TABEL KOMPARASI: LAMPUNG TENGAH VS KABUPATEN/KOTA LAIN ===")
komparasi = df_final.groupby('wilayah_fokus').agg(
    total_listing=('listing_id', 'count'),
    listing_unik=('listing_id', 'nunique'),
    rata_rata_harga=('harga', 'mean'),
    median_harga=('harga', 'median'),
    harga_termurah=('harga', 'min'),
    harga_termahal=('harga', 'max'),
    rata_rata_rating=('rating', 'mean'),
    total_ulasan=('jumlah_review', 'sum'),
    persen_punya_ulasan=('has_reviews', lambda x: round(x.mean() * 100, 2))
).reset_index()
display(komparasi)

# 3. Distribusi Price Tier per Wilayah Fokus
print("\n=== DISTRIBUSI PRICE TIER BERDASARKAN WILAYAH FOKUS ===")
ct_tier = pd.crosstab(df_final['wilayah_fokus'], df_final['price_tier'], margins=True)
display(ct_tier)

# 4. Top 10 Listing Terbaik di Lampung (Rating 5.0 & Ulasan Terbanyak)
print("\n=== TOP 10 LISTING DENGAN RATING & ULASAN TERTINGGI (LISTING UNIK) ===")
top10_unique = (
    df_final[df_final['rating'].notna()]
    .groupby('listing_id')
    .agg(
        nama=('nama', 'first'),
        tipe_properti=('tipe_properti', 'first'),
        kabupaten_kota=('kabupaten_kota', 'first'),
        harga=('harga', 'first'),
        rating=('rating', 'first'),
        jumlah_review=('jumlah_review', 'first')
    )
    .sort_values(by=['rating', 'jumlah_review', 'harga'], ascending=[False, False, True])
    .head(10)
    .reset_index()
)
display(top10_unique)


## 5. Load & Export (Persiapan Google BigQuery)

Pada tahap akhir ETL:
1. Kita mengekspor dataset final bersih ke berkas CSV berstandar UTF-8 tanpa index di `output/airbnb_lampung_clean.csv`.
2. Kita membuat skema tabel BigQuery berformat JSON di `output/bigquery_schema.json` yang siap digunakan melalui BigQuery CLI (`bq load`) atau Google Cloud Console.


In [ ]:
# Tentukan folder output
output_dir = '../output' if os.path.exists('../output') else 'output'
os.makedirs(output_dir, exist_ok=True)

# 1. Simpan CSV Final Bersih
clean_csv_path = os.path.join(output_dir, 'airbnb_lampung_clean.csv')
df_final.to_csv(clean_csv_path, index=False, encoding='utf-8')
print(f"[OK] File CSV bersih berhasil disimpan di: {clean_csv_path}")

# 2. Definisikan dan Simpan Skema BigQuery JSON
bq_schema = [
    {"name": "listing_id", "type": "STRING", "mode": "REQUIRED", "description": "ID unik listing Airbnb yang diekstrak dari URL tautan"},
    {"name": "nama", "type": "STRING", "mode": "REQUIRED", "description": "Nama lengkap listing akomodasi pada platform Airbnb"},
    {"name": "tipe_properti", "type": "STRING", "mode": "NULLABLE", "description": "Tipe akomodasi (Home, Apartment, Hotel, Guesthouse, Villa, dll.)"},
    {"name": "lokasi_spesifik", "type": "STRING", "mode": "NULLABLE", "description": "Lokasi spesifik, nama kecamatan, atau area properti"},
    {"name": "judul_listing", "type": "STRING", "mode": "NULLABLE", "description": "Judul deskriptif properti hasil parsing nama"},
    {"name": "kabupaten_kota", "type": "STRING", "mode": "REQUIRED", "description": "Nama resmi kabupaten atau kota di Provinsi Lampung"},
    {"name": "wilayah_fokus", "type": "STRING", "mode": "REQUIRED", "description": "Klasifikasi komparasi: Kabupaten Lampung Tengah vs Kabupaten/Kota Lain (Lampung)"},
    {"name": "harga", "type": "INT64", "mode": "REQUIRED", "description": "Harga sewa menginap per malam dalam mata uang Rupiah (IDR)"},
    {"name": "rating", "type": "FLOAT64", "mode": "NULLABLE", "description": "Skor rating ulasan pengguna Airbnb (skala 1.0 - 5.0)"},
    {"name": "rating_kategori", "type": "STRING", "mode": "REQUIRED", "description": "Kategori performa ulasan listing akomodasi"},
    {"name": "is_new_listing", "type": "BOOL", "mode": "REQUIRED", "description": "Penanda listing berstatus baru (New di Airbnb)"},
    {"name": "jumlah_review", "type": "INT64", "mode": "REQUIRED", "description": "Jumlah ulasan tamu yang diterima listing"},
    {"name": "has_reviews", "type": "BOOL", "mode": "REQUIRED", "description": "Penanda apakah listing telah memiliki ulasan tamu"},
    {"name": "price_tier", "type": "STRING", "mode": "REQUIRED", "description": "Segmentasi tarif sewa berbasis kuartil: Budget, Mid-Range, Premium"},
    {"name": "link", "type": "STRING", "mode": "REQUIRED", "description": "Tautan URL langsung menuju halaman listing di situs Airbnb"}
]

schema_file_path = os.path.join(output_dir, 'bigquery_schema.json')
with open(schema_file_path, 'w', encoding='utf-8') as f:
    json.dump(bq_schema, f, indent=2, ensure_ascii=False)
print(f"[OK] Skema BigQuery JSON berhasil disimpan di: {schema_file_path}")


## 6. Panduan Pemuatan ke Google BigQuery (CLI & UI)

### Cara Memuat Data ke Google BigQuery:

#### Opsi A: Menggunakan Google Cloud SDK (`bq` CLI)
Jalankan perintah berikut di terminal local/Cloud Shell:
```bash
# 1. Buat Dataset jika belum ada
bq --location=asia-southeast2 mk -d \
    --description "Dataset Analisis Pasar Airbnb Provinsi Lampung" \
    bigdata-airbnb-lampung:airbnb_lampung_ds

# 2. Muat berkas CSV menggunakan skema JSON yang telah diekspor
bq load \
    --source_format=CSV \
    --skip_leading_rows=1 \
    --replace \
    bigdata-airbnb-lampung:airbnb_lampung_ds.listings_clean \
    ./output/airbnb_lampung_clean.csv \
    ./output/bigquery_schema.json
```

#### Opsi B: Menggunakan Google Cloud Console (Web UI)
1. Buka console Google Cloud BigQuery: `https://console.cloud.google.com/bigquery`
2. Pilih project `bigdata-airbnb-lampung` dan buat dataset `airbnb_lampung_ds`.
3. Klik tombol **Create Table** di dalam dataset `airbnb_lampung_ds`.
4. Sumber: Pilih **Upload**, unggah file `airbnb_lampung_clean.csv`.
5. Format: **CSV**.
6. Nama Tabel: `listings_clean`.
7. Schema: Aktifkan mode **Edit as text**, lalu salin dan tempel isi berkas `bigquery_schema.json`.
8. Klik **Create Table**.
